### **Representación visual, BoVW, CNN features, region features y alineamiento imagen-texto**

Este cuaderno está diseñado como **repaso previo** de los siguientes temas sobre:
- modelos tempranos de alineamiento visual-semántico,
- representaciones iniciales de texto e imagen,
- antecedentes del aprendizaje multimodal.

A diferencia de una introducción básica, aquí se incorporan:
- formulación vectorial explícita,
- cuantización visual y histogramas,
- features globales y por regiones,
- espacios compartidos imagen-texto,
- similitud coseno,
- **margin ranking loss**,
- **InfoNCE/in-batch contrastive loss**,
- evaluación de retrieval mediante **Recall@K**.

#### **Qué debemos consolidad al terminar**

1. Qué diferencia hay entre una representación visual local, global y regional.
2. Cómo una imagen puede convertirse en un histograma BoVW o en un embedding CNN.
3. Cómo texto e imagen se proyectan a un espacio común.
4. Qué problema optimiza una margin ranking loss y qué cambia al usar una loss contrastiva tipo CLIP.
5. Cómo evaluar un sistema de alineamiento con una matriz de similitud y Recall@K.

#### **1. Importaciones y configuración**
Todo el notebook está diseñado para ejecutarse sin descargas externas obligatorias.

In [ ]:
# Jupyter/ otebook
%pip install scikit-image

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from skimage import data, color, transform
from skimage.util import view_as_windows
from skimage.feature import hog

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

#### **2. Marco conceptual: representaciones visuales**

Sea una imagen $I$. En visión por computador, una decisión central es **cómo codificarla** en una forma que preserve información relevante para la tarea.  
Formalmente, una representación visual es una transformación:

$$
I \longmapsto z(I)
$$

donde $z(I)$ puede ser:
- un único vector global,
- una colección de vectores locales,
- un histograma de patrones visuales,
- o una secuencia/conjunto de embeddings regionales.

La elección de esta representación no es neutra: determina qué tipo de estructura visual puede capturar el modelo y qué tipo de alineamiento puede realizar después con el lenguaje.

##### **2.1 Representación global**

La forma más simple consiste en resumir toda la imagen en un único vector:

$$
v = E_{img}(I) \in \mathbb{R}^d
$$

donde $E_{img}$ es un codificador visual.  

Este vector puede obtenerse de varias maneras:

- aplanando pixeles y aplicando una proyección,
- usando un descriptor global diseñado manualmente,
- extrayendo la activación de una capa intermedia o final de una CNN,
- usando la salida agregada de un Vision Transformer.

##### **Interpretación**
La representación global intenta condensar el contenido visual total de la imagen en un solo embedding.  
Esto es útil cuando:
- la tarea depende del contenido general,
- la imagen contiene un objeto dominante,
- o basta una semántica de alto nivel.

##### **Ventajas**
- compacta y eficiente,
- fácil de comparar con similitud coseno o producto interno,
- adecuada para clasificación o retrieval global.

##### **Limitaciones**
- pierde estructura espacial fina,
- no distingue explícitamente entre distintas regiones u objetos,
- puede ser insuficiente cuando el texto describe partes específicas de la imagen.

En términos geométricos, dos imágenes $I_1$ e $I_2$ se consideran visualmente cercanas si sus embeddings globales $v_1$ y $v_2$ están próximos según alguna métrica, por ejemplo:

$$
\operatorname{sim}(I_1,I_2)=
\frac{v_1^\top v_2}{\|v_1\|\|v_2\|}
$$

##### **2.2 Representación local**

En lugar de un solo vector global, puede representarse una imagen mediante una colección de descriptores asociados a parches o regiones:

$$
x_r = \phi(I_r), \quad r=1,\dots,R
$$

donde:
- $I_r$ es el parche o región $r$,
- $\phi$ es un extractor local de rasgos,
- $x_r \in \mathbb{R}^p$ es el descriptor resultante.

Históricamente, estos descriptores locales fueron fundamentales porque permitían capturar:
- bordes,
- orientación,
- textura,
- patrones repetitivos,
- y estructura visual parcial.

Ejemplos clásicos incluyen:
- SIFT,
- SURF,
- HOG,
- descriptores de color o textura,
- y, más adelante, activaciones convolucionales locales.

##### **Interpretación**
La imagen deja de verse como una unidad monolítica y pasa a entenderse como un conjunto de observaciones locales.  
Esto permite representar mejor:
- escenas complejas,
- múltiples objetos,
- variaciones de posición,
- y correspondencias parciales entre imagen y texto.

##### **Ventajas**
- mayor sensibilidad a estructura local,
- más robustez a pequeñas transformaciones,
- útil para construir representaciones agregadas posteriores.

##### **Limitaciones**
- produce un conjunto variable de vectores,
- requiere una estrategia de agregación o cuantización,
- no define por sí sola una representación global comparable.

##### **2.3 Representación BoVW**

La representación **Bag of Visual Words (BoVW)** traslada a visión una idea análoga a Bag of Words en texto.  
A partir de muchos descriptores locales extraídos del conjunto de entrenamiento, se aprende un vocabulario visual:

$$
C = \{c_1,\dots,c_K\}, \qquad c_k \in \mathbb{R}^p
$$

donde cada $c_k$ es un centroide o "palabra visual" aprendido típicamente con *k-means*.

Luego, cada descriptor local $x_r$ se cuantiza asignándolo al centroide más cercano:

$$
q(x_r) = \arg\min_{k \in \{1,\dots,K\}} \|x_r - c_k\|_2^2
$$

y la imagen se representa mediante un histograma:

$$
h_I[k] = \sum_{r=1}^{R} \mathbf{1}\big(q(x_r)=k\big), \qquad k=1,\dots,K
$$

A menudo este histograma se normaliza:

$$
\tilde{h}_I = \frac{h_I}{\|h_I\|_1}
\quad \text{o} \quad
\tilde{h}_I = \frac{h_I}{\|h_I\|_2}
$$

##### **Interpretación**
BoVW no conserva la ubicación exacta de cada patrón, sino **qué tipos de patrones aparecen y con qué frecuencia**.  
Por eso produce una representación:
- global,
- vectorial,
- comparable entre imágenes,
- pero esencialmente **orderless**.

##### **Ventajas**
- convierte una colección variable de descriptores locales en un vector fijo,
- permite usar herramientas estándar de clasificación y recuperación,
- fue una de las primeras formas exitosas de mapear visión a un espacio distribuido.

##### **Limitaciones**
- la cuantización dura puede introducir error,
- pierde gran parte de la estructura espacial,
- depende de la calidad del descriptor local y del vocabulario visual,
- no aprende necesariamente una representación semántica de alto nivel.

En variantes más sofisticadas, la cuantización puede hacerse de forma:
- **suave** (*soft assignment*),
- con codificación dispersa,
- o con métodos como VLAD y Fisher Vectors, que preservan más información que un histograma puro.

##### **2.4 Representación por regiones**

En muchos sistemas visión-lenguaje, especialmente antes de los encoders globales contrastivos modernos, la imagen se representaba como un conjunto de embeddings regionales:

$$
V_I = [v_1, v_2, \dots, v_M], \quad v_m \in \mathbb{R}^d
$$

donde cada $v_m$ resume una región relevante de la imagen.  
Estas regiones pueden provenir de:
- una rejilla fija,
- propuestas de regiones,
- detectores de objetos,
- o mecanismos de atención.

Aquí ya no se asume que toda la imagen deba comprimirse en un único vector.  
En cambio, el modelo puede operar con una estructura más elaborada:

$$
V_I \in \mathbb{R}^{M \times d}
$$

donde $M$ es el número de regiones y $d$ la dimensión del embedding regional.

##### **Interpretación**
Esta representación es especialmente útil cuando el texto hace referencia a:
- objetos concretos,
- atributos locales,
- relaciones espaciales,
- o partes específicas de una escena.

Por ejemplo, una oración como:  *"a small dog next to a red ball"*  
difícilmente puede alinearse bien con un único vector global si queremos capturar correspondencias finas entre palabras y contenido visual.

##### **Ventajas**
- permite alineamiento más fino palabra-región o frase-región,
- preserva más estructura semántica local,
- es una base natural para atención visual-semántica.

##### **Limitaciones**
- aumenta el costo computacional,
- requiere decidir cómo obtener y filtrar regiones,
- introduce el problema adicional de agregación, selección o atención.

##### **2.5 Comparación entre niveles de representación**

Podemos resumir la progresión así:

$$
\text{pixeles} \rightarrow \text{descriptores locales} \rightarrow \text{BoVW / agregación} \rightarrow \text{embeddings CNN globales} \rightarrow \text{region features}
$$

Cada etapa responde a una pregunta distinta:

- **Representación global:** ¿cómo resumir toda la imagen en un vector?
- **Representación local:** ¿cómo describir estructura visual en parches?
- **BoVW:** ¿cómo convertir muchos descriptores locales en un vector fijo?
- **Representación por regiones:** ¿cómo preservar granularidad semántica útil para alineamiento fino?

##### **2.6 Relevancia para alineamiento imagen-texto**

Esta discusión es central para aprendizaje multimodal porque el texto no se alinea con "la imagen en bruto", sino con alguna representación visual:

$$
I \longmapsto v
\quad \text{o} \quad
I \longmapsto \{v_1,\dots,v_M\}
$$

Solo después tiene sentido plantear un espacio compartido con el lenguaje:

$$
z_v = W_v v, \qquad z_t = W_t t
$$

o, en el caso regional,

$$
z_{v_m} = W_v v_m, \qquad m=1,\dots,M
$$

De este modo, la calidad del alineamiento visual-semántico depende en gran medida de **qué representación visual se eligió** y **qué información decidió preservar**.

#### **3. Imágenes de ejemplo**

In [ ]:
images = {
    "astronaut": data.astronaut(),
    "coffee": data.coffee(),
    "camera": data.camera(),
    "coins": data.coins()
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

#### **4. Bag of Visual Words (BoVW)**

BoVW traduce una imagen a un histograma de patrones visuales recurrentes.  
El pipeline canónico es:
1. extraer descriptores locales,
2. aprender un código visual con *k-means*,
3. asignar cada descriptor a una palabra visual,
4. construir un histograma.

In [ ]:
def extract_gray_patches(image, patch_size=16, step=8, resize_to=(128, 128)):
    if image.ndim == 3:
        image = color.rgb2gray(image)
    image = transform.resize(image, resize_to, anti_aliasing=True)
    windows = view_as_windows(image, (patch_size, patch_size), step=step)
    patches = windows.reshape(-1, patch_size, patch_size)
    return patches

def hog_descriptor(patch):
    return hog(
        patch,
        orientations=8,
        pixels_per_cell=(8, 8),
        cells_per_block=(1, 1),
        feature_vector=True
    )

patches = extract_gray_patches(images["camera"])
descs = np.array([hog_descriptor(p) for p in patches[:8]])

print("Número total de parches:", len(patches))
print("Dimensión del descriptor HOG:", descs.shape[1])

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, patch in zip(axes.ravel(), patches[:8]):
    ax.imshow(patch, cmap="gray")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
all_descs = []
for img in images.values():
    local_patches = extract_gray_patches(img, patch_size=16, step=8)
    local_descs = [hog_descriptor(p) for p in local_patches[:100]]
    all_descs.extend(local_descs)

all_descs = np.array(all_descs)
print("Matriz de descriptores:", all_descs.shape)

K = 24
visual_vocab = MiniBatchKMeans(n_clusters=K, random_state=42, batch_size=128, n_init=10)
visual_vocab.fit(all_descs)

print("Forma del vocabulario visual:", visual_vocab.cluster_centers_.shape)

In [ ]:
def bovw_encode(image, codebook, patch_size=16, step=8):
    patches = extract_gray_patches(image, patch_size=patch_size, step=step)
    descs = np.array([hog_descriptor(p) for p in patches])
    assignments = codebook.predict(descs)
    hist, _ = np.histogram(assignments, bins=np.arange(codebook.n_clusters + 1))
    hist = hist.astype(np.float32)
    hist /= hist.sum() + 1e-8
    return hist

bovw = {name: bovw_encode(img, visual_vocab) for name, img in images.items()}

fig, axes = plt.subplots(4, 1, figsize=(10, 8))
for ax, (name, hist) in zip(axes, bovw.items()):
    ax.bar(np.arange(len(hist)), hist)
    ax.set_title(f"BoVW - {name}")
    ax.set_ylabel("freq.")
plt.xlabel("palabra visual")
plt.tight_layout()
plt.show()

In [ ]:
names = list(bovw.keys())
H = np.vstack([bovw[n] for n in names])
sim_bovw = cosine_similarity(H)

pd.DataFrame(np.round(sim_bovw, 3), index=names, columns=names)

##### **Comentario**

La representación **Bag of Visual Words (BoVW)** ocupó un lugar importante en la evolución de la visión por computador porque ofreció una forma sistemática de transformar una imagen, que originalmente es una estructura espacial compleja en un **vector fijo y comparable**.

Su importancia puede entenderse en varios niveles:

##### **1. Convierte información visual local en una representación global**
BoVW parte de descriptores locales extraídos de parches o puntos de interés.  
Cada uno de esos descriptores codifica información de bajo o medio nivel, como:
- bordes,
- orientación,
- textura,
- patrones repetitivos.

Luego, al cuantizarlos contra un vocabulario visual y acumular sus ocurrencias en un histograma, la imagen queda representada por un vector de dimensión fija.  
Eso fue crucial porque permitió aplicar sobre imágenes muchas herramientas ya conocidas de aprendizaje estadístico:
- clasificación lineal,
- SVM,
- clustering,
- recuperación basada en similitud.

En otras palabras, BoVW resolvió parcialmente el problema de pasar de una estructura visual irregular y variable a un objeto algebraicamente manejable.

##### **2. Desacopla extracción local y representación global**
Una ventaja metodológica importante es que BoVW separa dos decisiones distintas:

- **cómo describir localmente la imagen**  
  (por ejemplo, con SIFT, HOG u otros descriptores),

- **cómo agregar esa información a nivel global**  
  (por ejemplo, mediante cuantización y conteo).

Ese desacoplamiento hizo posible combinar distintos descriptores locales con distintos vocabularios visuales y comparar su efecto sobre la tarea final.  
Desde un punto de vista histórico, esto fue útil porque permitió estudiar el problema representacional de manera modular:
- primero diseñar buenos descriptores,
- luego diseñar una buena estrategia de codificación global.

##### **3. Introduce en visión una lógica distribucional análoga a Bag of Words en NLP**
BoVW establece una analogía fuerte con las representaciones distribucionales clásicas del texto:

- en NLP, un documento puede representarse por conteos de palabras,
- en visión, una imagen puede representarse por conteos de "palabras visuales".

La idea compartida es que una estructura compleja puede resumirse por la **distribución de patrones elementales** que contiene.  
Esto conecta BoVW con una tradición más amplia de representación basada en:
- conteos,
- coocurrencias,
- histogramas,
- y espacios vectoriales comparables.

Por eso BoVW es conceptualmente útil por que ayuda a ver que texto e imagen, aunque provienen de modalidades distintas, pueden mapearse a espacios vectoriales construidos con lógicas parcialmente análogas.


##### **4. Fue un paso importante antes de las representaciones profundas**
Antes del dominio de CNNs y transformers visuales, BoVW fue una de las estrategias más eficaces para obtener representaciones visuales útiles.  
Aunque hoy se considera una técnica "clásica", históricamente fue importante porque mostró que:
- la imagen podía representarse distribucionalmente,
- esa representación podía funcionar razonablemente bien en tareas reales,
- y era posible comparar imágenes mediante métricas vectoriales estándar.

Desde una perspectiva pedagógica, BoVW ayuda a entender el paso intermedio entre:
- descriptores manuales,
- agregación estadística,
- y representaciones profundas aprendidas.

##### **5. Limitaciones conceptuales de BoVW**

A pesar de su importancia, BoVW tiene restricciones importantes que explican por qué fue superado por enfoques posteriores.

**a. Pierde gran parte de la estructura espacial**

El histograma final indica **qué patrones aparecen** y **con qué frecuencia**, pero no conserva bien:
- dónde aparecen,
- en qué relación espacial están,
- ni cómo se organizan entre sí.

Dos imágenes con histogramas similares pueden tener configuraciones espaciales muy distintas.  
Esto limita su capacidad para modelar:
- composición visual,
- relaciones entre objetos,
- y correspondencias finas con el lenguaje.

**b. Depende de cuantización dura**

La asignación de cada descriptor local a una sola palabra visual:

$$
q(x_r)=\arg\min_k \|x_r-c_k\|_2^2
$$

impone una decisión discreta que puede ser inestable.  
Si un descriptor está cerca de varios centroides, la cuantización dura puede introducir error y pérdida de información.

En otras palabras, la representación final depende bastante de:
- la calidad del vocabulario visual,
- el número de centroides,
- y la geometría del espacio de descriptores.

**c. No aprende la representación de extremo a extremo**
En BoVW, las etapas del pipeline suelen estar separadas:
- extracción de descriptores,
- clustering,
- asignación,
- conteo,
- clasificación o retrieval.

Esto significa que la representación visual no se optimiza directamente con respecto a la tarea final.  
Los modelos profundos posteriores resultaron más potentes precisamente porque permiten aprender representaciones visuales ajustadas por gradiente al objetivo de entrenamiento.

**d. Tiene capacidad semántica limitada**

BoVW puede capturar regularidades estadísticas de patrones visuales, pero no necesariamente produce una representación de alto nivel alineada con conceptos semánticos complejos.  
Eso lo vuelve menos adecuado para tareas que exigen:
- grounding fino,
- composición visual,
- alineamiento palabra-región,
- o razonamiento multimodal.

##### **6. Comentario**
Desde una perspectiva histórica, BoVW puede entenderse como una etapa intermedia entre:

$$
\text{descriptores locales manuales}
\,\rightarrow\,
\text{agregación vectorial}
\,\rightarrow\,
\text{representaciones profundas aprendidas}
$$

Por eso sigue siendo útil estudiarlo: no porque sea la técnica dominante hoy, sino porque ayuda a entender cómo la visión por computador fue construyendo el camino hacia representaciones más abstractas y compatibles con modelos multimodales posteriores.

#### **5. CNN features: representación global aprendida**

En contraste con enfoques clásicos como **BoVW**, donde la imagen se representa mediante histogramas de patrones visuales cuantizados, una red convolucional aprende directamente una función de representación:

$$
v = E_{img}(I)
$$

donde:
- $I$ es la imagen de entrada,
- $E_{img}$ es el codificador visual aprendido,
- $v \in \mathbb{R}^d$ es un **embedding visual global** de dimensión fija.

La idea central es que una CNN no depende de un vocabulario visual construido por separado, sino que aprende jerárquicamente sus propias representaciones a partir de los datos.  
En términos conceptuales, las distintas capas suelen capturar niveles de abstracción diferentes:

- **capas iniciales**: bordes, gradientes y texturas simples,
- **capas intermedias**: patrones locales más complejos y partes de objetos,
- **capas profundas**: rasgos semánticos de alto nivel más cercanos a categorías visuales.

De este modo, el embedding global $v$ puede interpretarse como una compresión aprendida del contenido visual de la imagen, útil para tareas como:
- clasificación,
- recuperación visual,
- comparación entre imágenes,
- y, en nuestro contexto, **alineamiento imagen-texto**.

##### **Ventajas frente a BoVW**
Las CNN features introducen varias mejoras respecto a representaciones clásicas basadas en descriptores manuales:

- aprenden la representación de forma **extremo a extremo**,
- capturan patrones visuales más abstractos y semánticos,
- evitan la cuantización dura de descriptores locales,
- producen embeddings más adecuados para comparación y transferencia.

##### **Limitaciones**
Aun así, una representación global también tiene limitaciones:
- comprime toda la imagen en un solo vector,
- puede perder información espacial fina,
- no distingue explícitamente entre múltiples objetos o regiones relevantes.

Por eso, en tareas de alineamiento fino entre lenguaje e imagen, muchas veces se complementa o reemplaza por **region features** o mecanismos de atención.

##### **Relevancia para aprendizaje multimodal**
En modelos tempranos de alineamiento visual-semántico, la representación visual ya no se construye solo con descriptores manuales agregados, sino también con embeddings aprendidos por redes neuronales.  
Eso permite plantear un espacio compartido entre imagen y texto:

$$
z_v = W_v v, \qquad z_t = W_t t
$$

donde el embedding visual global $v$ y el embedding textual $t$ pueden proyectarse a un mismo espacio para medir similitud.

##### **Nota sobre este cuaderno**
Aquí usaremos una CNN pequeña sobre el dataset `digits` no porque sea un benchmark realista de visión-lenguaje, sino porque permite ilustrar de forma controlada:

1. cómo se obtiene un embedding visual aprendido,
2. cómo ese embedding puede compararse con otros vectores,
3. y cómo luego puede reutilizarse en un esquema sencillo de alineamiento imagen-texto.

In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)

label_names = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img, label in zip(axes.ravel(), X[:10], y[:10]):
    ax.imshow(img, cmap="gray")
    ax.set_title(label_names[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Shape del dataset:", X.shape)

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images[:, None, :, :], dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(DigitsDataset(X_train, y_train), batch_size=128, shuffle=True)
test_loader  = DataLoader(DigitsDataset(X_test, y_test), batch_size=256, shuffle=False)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, feat_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc_feat = nn.Linear(32 * 2 * 2, feat_dim)
        self.fc_cls = nn.Linear(feat_dim, 10)

    def forward(self, x, return_features=False):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.flatten(1)
        feat = self.fc_feat(x)
        if return_features:
            return feat
        logits = self.fc_cls(F.relu(feat))
        return logits

cnn = SmallCNN(feat_dim=64)
cnn

In [ ]:
def train_cnn(model, train_loader, test_loader, num_epochs=6, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(xb)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                pred = model(xb).argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += len(yb)

        print(f"Epoca {epoch+1:02d} | loss={total_loss/len(train_loader.dataset):.4f} | acc={correct/total:.4f}")

train_cnn(cnn, train_loader, test_loader, num_epochs=6, lr=1e-3)

In [ ]:
cnn.eval()
with torch.no_grad():
    sample_x = torch.tensor(X_test[:12, None, :, :], dtype=torch.float32)
    feat = cnn(sample_x, return_features=True).numpy()

print("Forma de las CNN features:", feat.shape)

proj = PCA(n_components=2, random_state=42).fit_transform(feat)
plt.figure(figsize=(6, 4))
plt.scatter(proj[:, 0], proj[:, 1], c=y_test[:12], s=80)
for i, lab in enumerate(y_test[:12]):
    plt.text(proj[i, 0], proj[i, 1], label_names[lab], fontsize=8)
plt.title("CNN features proyectadas con PCA")
plt.show()

#### **6. Region features**

En muchos sistemas visión-lenguaje tempranos y de transición no se representaba la imagen únicamente mediante un vector global, sino como un **conjunto de vectores por región**.  
La motivación es clara: un único embedding global puede resumir bien el contenido general de la imagen, pero suele ser insuficiente cuando el texto se refiere a **objetos específicos, atributos locales o relaciones espaciales**.

Formalmente, una imagen $I$ puede representarse como una colección de embeddings regionales:

$$
V_I = [v_1,\dots,v_M], \qquad v_m = E_{reg}(R_m)
$$

donde:
- $R_m$ es la región $m$-ésima de la imagen,
- $E_{reg}$ es un codificador visual aplicado a cada región,
- $v_m \in \mathbb{R}^d$ es el embedding regional resultante,
- $M$ es el número de regiones consideradas.

También puede verse como una matriz:

$$
V_I \in \mathbb{R}^{M \times d}
$$

donde cada fila corresponde a una región y cada columna a una dimensión del embedding.

##### **¿Qué es una región?**

Una región puede definirse de varias maneras:
- una **celda de una rejilla fija**,
- una **propuesta de región** generada por algún algoritmo,
- una **caja delimitadora** asociada a un objeto detectado,
- o, en enfoques más modernos, una **unidad latente** inducida por atención.

La elección del conjunto de regiones determina qué estructura local puede capturar el modelo.

##### **Interpretación conceptual**
La representación por regiones introduce una hipótesis más detallada que la representación global:

- la imagen no se trata como una sola unidad semántica,
- sino como una composición de partes potencialmente alineables con fragmentos del lenguaje.

Esto es especialmente importante cuando una oración describe:
- múltiples objetos,
- atributos visuales localizados,
- interacciones entre entidades,
- o detalles que no dominan la imagen completa.

Por ejemplo, una descripción como:

> *a small dog next to a red ball*

es difícil de alinear correctamente con un único vector global, porque el texto menciona al menos dos entidades y una relación espacial.  
En cambio, una representación por regiones permite que diferentes partes de la imagen contribuyan de manera distinta al significado total.

##### **Ventajas de los region features**
Las representaciones regionales son valiosas porque:

1. **preservan granularidad semántica local**,  
   ya que cada región puede corresponder a un objeto, una parte de objeto o una zona visualmente distintiva.

2. **permiten alineamiento más fino con el lenguaje**,  
   por ejemplo palabra-región o frase-región.

3. **facilitan mecanismos de atención visual-semántica**,  
   donde el modelo aprende qué regiones son más relevantes para una consulta textual.

4. **mejoran la interpretabilidad**,  
   ya que es más fácil inspeccionar qué regiones contribuyen a una decisión o predicción.

##### **Limitaciones**
Sin embargo, esta representación también introduce complejidad:

- el número de regiones $M$ puede variar o crecer mucho,
- no todas las regiones son informativas,
- puede haber redundancia entre regiones cercanas,
- se requiere una estrategia para:
  - seleccionar,
  - ponderar,
  - agregar,
  - o atender sobre las regiones.

Además, el rendimiento depende bastante de cómo se construyen esas regiones:
- una rejilla fija es simple, pero poco semántica,
- un detector de objetos es más informativo, pero introduce dependencia de un modelo adicional.

##### **Relación con el alineamiento imagen-texto**
Si el texto tiene embedding $t$, una representación global compara típicamente:

$$
s(I, x) = \operatorname{sim}(v, t)
$$

mientras que con regiones puede pensarse en comparaciones más finas, por ejemplo:

$$
s(I, x) = \max_{m=1,\dots,M} \operatorname{sim}(v_m, t)
$$

o en variantes donde la similitud se agrega con pesos de atención.  
Esto hace posible que el modelo no pregunte solo **"¿la imagen y el texto se parecen?"**, sino también **"¿qué parte de la imagen se parece más al contenido textual?"**.

##### **Comentario**

Los **region features** fueron especialmente importantes en la transición entre:
- representaciones visuales clásicas agregadas,
- modelos de embedding global imagen-texto,
- y sistemas con alineamiento más fino y atención visual-semántica.

Por eso son una pieza conceptual clave para entender la evolución desde:
- representaciones globales simples,
- hacia modelos capaces de grounding y correspondencia local entre visión y lenguaje.

In [ ]:
def grid_regions(image, rows=3, cols=3):
    if image.ndim == 2:
        image_rgb = np.stack([image, image, image], axis=-1)
    else:
        image_rgb = image.copy()

    h, w = image_rgb.shape[:2]
    rh, rw = h // rows, w // cols

    regions, boxes = [], []
    for r in range(rows):
        for c in range(cols):
            y1 = r * rh
            x1 = c * rw
            y2 = h if r == rows - 1 else (r + 1) * rh
            x2 = w if c == cols - 1 else (c + 1) * rw
            regions.append(image_rgb[y1:y2, x1:x2])
            boxes.append((x1, y1, x2, y2))
    return regions, boxes

img = images["astronaut"]
regions, boxes = grid_regions(img, rows=3, cols=3)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img)
for (x1, y1, x2, y2) in boxes:
    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=1.5))
ax.set_title("Grilla de regiones")
ax.axis("off")
plt.show()

In [ ]:
def region_descriptor(region):
    region_resized = transform.resize(region, (64, 64), anti_aliasing=True)
    if region_resized.ndim == 2:
        rgb_means = np.array([region_resized.mean()] * 3)
        gray = region_resized
    else:
        rgb_means = region_resized.mean(axis=(0, 1))
        gray = color.rgb2gray(region_resized)

    hog_feat = hog(
        gray,
        orientations=8,
        pixels_per_cell=(16, 16),
        cells_per_block=(1, 1),
        feature_vector=True
    )
    return np.concatenate([rgb_means, hog_feat])

region_feats = np.array([region_descriptor(r) for r in regions])
print("Shape de region features:", region_feats.shape)

#### **7. Espacio compartido imagen-texto**

Uno de los problemas fundamentales del aprendizaje multimodal temprano es que imagen y texto pertenecen, en principio, a espacios de representación distintos:

- la imagen se codifica a partir de información visual,
- el texto se codifica a partir de información lingüística.

Por eso, antes de compararlos, es necesario construir una representación común que haga posible medir compatibilidad semántica entre ambas modalidades.

Partimos de dos codificadores:

$$
v = E_{img}(I), \qquad t = E_{text}(x)
$$

donde:
- $I$ es la imagen,
- $x$ es el texto, palabra, frase u oración,
- $E_{img}$ produce una representación visual $v \in \mathbb{R}^{d_v}$,
- $E_{text}$ produce una representación textual $t \in \mathbb{R}^{d_t}$.

En general, $d_v$ y $d_t$ pueden ser diferentes, y además ambos vectores responden a estructuras muy distintas.  
Por ello, se introducen proyecciones hacia un **espacio compartido** de dimensión común $d$:

$$
z_v = W_v v,\qquad z_t = W_t t
$$

donde:
- $W_v \in \mathbb{R}^{d \times d_v}$,
- $W_t \in \mathbb{R}^{d \times d_t}$,
- $z_v, z_t \in \mathbb{R}^d$.

Estas proyecciones pueden interpretarse como transformaciones aprendidas que obligan a las dos modalidades a volverse comparables en términos geométricos.

##### **Interpretación conceptual**
La idea del espacio compartido no es simplemente "reducir dimensión", sino aprender una geometría en la que:

- una imagen y su texto correspondiente queden cercanos,
- una imagen y un texto no relacionado queden lejanos,
- la cercanía refleje compatibilidad semántica y no solo similitud superficial.

En otras palabras, no interesa únicamente representar bien cada modalidad por separado, sino alinear ambas en una estructura común.

##### Función de similitud
Una vez obtenidos los embeddings proyectados, la comparación puede hacerse con similitud coseno:

$$
s(I,x) = \cos(z_v, z_t)
$$

es decir,

$$
s(I,x)=\frac{z_v^\top z_t}{\|z_v\|\,\|z_t\|}
$$

Esta medida es especialmente natural cuando los embeddings se normalizan, porque entonces la comparación depende de la orientación relativa de los vectores y no de su magnitud.

##### **¿Por qué similitud coseno?**
La similitud coseno es útil porque:

- es invariante a escala,
- permite comparar embeddings de distinta norma,
- tiene una interpretación geométrica clara,
- fue ampliamente usada en modelos tempranos de recuperación y alineamiento visual-semántico.

Si $z_v$ y $z_t$ apuntan en direcciones parecidas, su coseno será alto si son ortogonales o poco compatibles, será bajo.

##### **Espacio compartido como problema de alineamiento**
El objetivo del entrenamiento no es solo obtener buenos vectores visuales y textuales, sino satisfacer una restricción de orden relativo del tipo:

$$
s(I,x^+) > s(I,x^-)
$$

donde:
- $x^+$ es un texto correcto para la imagen $I$,
- $x^-$ es un texto incorrecto o negativo.

En muchos modelos tempranos, esta idea se implementa con pérdidas de ranking o margen, de modo que el par positivo no solo sea más similar que el negativo, sino que lo sea por al menos cierta cantidad.

##### **Dos maneras de entender el espacio compartido**

**1. Como alineamiento semántico**
La representación compartida busca que la estructura semántica se preserve entre modalidades.  

Por ejemplo:
- una imagen de un perro y la palabra *dog* deberían quedar cerca,
- una imagen de un perro y la palabra *car* deberían quedar lejos.

**2. Como interfaz multimodal**
El espacio compartido también actúa como una interfaz entre modalidades distintas.  
Una vez que imagen y texto viven en el mismo espacio, es posible hacer tareas como:
- recuperación imagen-texto,
- recuperación texto-imagen,
- clasificación zero-shot,
- matching multimodal.

##### **Caso más general**
En versiones más avanzadas, el alineamiento no tiene por qué hacerse solo entre un embedding visual global y un embedding textual global.  
También puede hacerse entre:
- regiones visuales y palabras,
- regiones visuales y frases,
- secuencias visuales y secuencias textuales.

En ese caso, la idea de espacio compartido se mantiene, pero la comparación deja de ser estrictamente vector contra vector y pasa a involucrar:
- máximos sobre regiones,
- atención,
- agregación ponderada,
- o alineamiento secuencial.

##### **Comentario**

Esta formulación fue central en los primeros modelos de **visual-semantic embeddings**.  
Antes del auge de los modelos contrastivos modernos, muchos sistemas multimodales ya podían describirse con esta estructura conceptual:

1. codificar imagen,
2. codificar texto,
3. proyectar ambas modalidades,
4. medir similitud,
5. optimizar para acercar pares correctos y separar pares incorrectos.

Por eso, entender este esquema es entender el núcleo del alineamiento visual-semántico temprano.

##### **Resumen conceptual**
La lógica del espacio compartido puede resumirse así:

$$
I \xrightarrow{E_{img}} v \xrightarrow{W_v} z_v
\qquad\qquad
x \xrightarrow{E_{text}} t \xrightarrow{W_t} z_t
$$

y luego:

$$
s(I,x)=\operatorname{sim}(z_v,z_t)
$$

donde la calidad del sistema depende de tres cosas:
- la calidad del codificador visual,
- la calidad del codificador textual,
- y la calidad del alineamiento geométrico aprendido entre ambos.

#### **8. Dataset pequeño imagen-texto**

Usaremos el dataset `digits` como problema supervisado mínimo.

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images[:, None, :, :], dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

align_train_loader = DataLoader(ImageTextDataset(X_train, y_train), batch_size=128, shuffle=True)
align_test_loader  = DataLoader(ImageTextDataset(X_test, y_test), batch_size=256, shuffle=False)

In [ ]:
class DualEncoder(nn.Module):
    def __init__(self, cnn_backbone, txt_vocab_size=10, txt_dim=32, shared_dim=32):
        super().__init__()
        self.cnn_backbone = cnn_backbone
        self.img_proj = nn.Linear(64, shared_dim)

        self.txt_emb = nn.Embedding(txt_vocab_size, txt_dim)
        self.txt_proj = nn.Linear(txt_dim, shared_dim)

    def encode_image(self, x):
        feat = self.cnn_backbone(x, return_features=True)
        z = self.img_proj(feat)
        return F.normalize(z, dim=1)

    def encode_text(self, token_ids):
        t = self.txt_emb(token_ids)
        z = self.txt_proj(t)
        return F.normalize(z, dim=1)

    def forward(self, x, token_ids):
        return self.encode_image(x), self.encode_text(token_ids)

dual = DualEncoder(cnn, txt_vocab_size=10, txt_dim=32, shared_dim=32)
dual

#### **9. Similitud coseno y margin ranking loss**

Una vez que imagen y texto han sido proyectados a un espacio compartido, necesitamos dos cosas:

1. una **función de similitud** que diga qué tan compatibles son sus embeddings,
2. una **función de pérdida** que obligue al modelo a acercar pares correctos y separar pares incorrectos.

##### **9.1 Similitud coseno**

Sea $u, v \in \mathbb{R}^d$. La **similitud coseno** se define como:

$$
\cos(u,v)=\frac{u^\top v}{\|u\|\,\|v\|}
$$

donde:
- $u^\top v$ es el producto interno entre ambos vectores,
- $\|u\|$ y $\|v\|$ son sus normas euclidianas.

##### **Interpretación geométrica**
La similitud coseno mide el coseno del ángulo entre dos vectores:

- si $\cos(u,v)\approx 1$, los vectores apuntan en direcciones muy parecidas,
- si $\cos(u,v)\approx 0$, son aproximadamente ortogonales,
- si $\cos(u,v)<0$, apuntan en direcciones opuestas.

En modelos multimodales tempranos, esta medida es especialmente útil porque no depende directamente de la magnitud de los embeddings, sino de su **alineación direccional**.

##### **Caso con embeddings normalizados**
Si los embeddings se normalizan a norma 1, es decir,

$$
\|u\| = \|v\| = 1,
$$

entonces la similitud coseno se reduce a:

$$
\cos(u,v)=u^\top v
$$

Esto simplifica mucho el cálculo y hace natural interpretar la similitud como un producto interno entre vectores unitarios.

##### **¿Por qué usar coseno y no distancia euclidiana?**
En espacios de embeddings, la norma puede variar por razones que no siempre son semánticamente relevantes.  
La similitud coseno es preferida porque:

- es invariante a escala,
- enfatiza orientación semántica más que magnitud,
- funciona bien en tareas de retrieval,
- y fue ampliamente usada en visual-semantic embeddings tempranos.

##### **Aplicación al caso imagen-texto**
Si $z_v$ es el embedding de una imagen y $z_t$ el embedding de un texto, la compatibilidad entre ambos puede definirse como:

$$
s(I,x)=\cos(z_v,z_t)
$$

Así, un buen sistema de alineamiento debería cumplir que:
- la similitud entre una imagen y su texto correcto sea alta,
- la similitud entre esa imagen y un texto incorrecto sea menor.

##### 9.2 *Margin ranking loss*

Una forma clásica de entrenar este alineamiento es mediante una **pérdida de ranking con margen**, que compara un par positivo con un par negativo.

Sea:
- $x^+$: texto correcto para la imagen $I$,
- $x^-$: texto incorrecto para la imagen $I$.

Entonces la pérdida puede escribirse como:

$$
\mathcal{L}=\max\big(0,\, m - s(I,x^+) + s(I,x^-)\big)
$$

donde:
- $s(I,x^+)$ es la similitud del par positivo,
- $s(I,x^-)$ es la similitud del par negativo,
- $m > 0$ es el **margen** deseado entre ambos.

##### **Interpretación**
La pérdida exige que el par correcto sea más similar que el incorrecto por al menos un margen $m$:

$$
s(I,x^+) \ge s(I,x^-) + m
$$

Si esa condición ya se cumple, entonces:

$$
\mathcal{L}=0
$$

y no hay penalización.  
Si no se cumple, el modelo recibe un gradiente que empuja:
- a **aumentar** la similitud del positivo,
- y/o a **disminuir** la del negativo.

##### **Lectura geométrica**
La margin ranking loss no solo busca que el positivo esté "más cerca" que el negativo, sino que exista una **separación mínima** entre ambos.  
Esto evita soluciones débiles donde el modelo apenas distingue entre pares correctos e incorrectos.

##### **Papel del margen $m$**
El hiperparámetro $m$ controla cuán estricta es la separación deseada:

- si $m$ es muy pequeño, el modelo puede aprender una separación débil
- si $m$ es muy grande, el problema puede volverse demasiado exigente e inestable.

Por tanto, $m$ regula el compromiso entre:
- facilidad de optimización,
- y fuerza de separación semántica.

##### **Ejemplo intuitivo**
Supongamos que:
- $s(I,x^+) = 0.80$
- $s(I,x^-) = 0.65$
- $m = 0.20$

Entonces:

$$
\mathcal{L}=\max(0,\,0.20 - 0.80 + 0.65)=\max(0,0.05)=0.05
$$

La pérdida es positiva porque el positivo no supera al negativo por un margen suficiente.

En cambio, si:
- $s(I,x^+) = 0.90$
- $s(I,x^-) = 0.50$
- $m = 0.20$

entonces:

$$
\mathcal{L}=\max(0,\,0.20 - 0.90 + 0.50)=\max(0,-0.20)=0
$$

Aquí el modelo ya separa correctamente ambos pares.


##### **9.3 ¿Qué aprende realmente el modelo?**

La combinación de similitud coseno y margin ranking loss induce una geometría en el espacio compartido donde:

- pares imagen-texto correctos se agrupan,
- pares incorrectos se separan,
- y la separación no es arbitraria, sino regulada por el margen.

Desde una perspectiva conceptual, este es el núcleo de muchos modelos tempranos de alineamiento visual-semántico:
1. codificar imagen,
2. codificar texto,
3. proyectar ambos a un espacio común,
4. comparar con una función de similitud,
5. optimizar una pérdida de ranking.

##### **9.4 Limitaciones**

Aunque esta formulación fue muy importante, también tiene límites:

- depende de cómo se seleccionan los negativos
- un solo negativo puede ser poco desafiante
- no siempre aprovecha toda la estructura del lote
- puede ser menos eficiente que pérdidas contrastivas modernas con múltiples negativos.

Por eso, históricamente esta familia de pérdidas dio paso a objetivos más fuertes, como **InfoNCE** o pérdidas contrastivas tipo **CLIP**.  
Aun así, la margin ranking loss sigue siendo una pieza fundamental para entender el aprendizaje multimodal temprano.

##### **9.5 Resumen conceptual**

La lógica puede resumirse así:

$$
z_v = W_v E_{img}(I), \qquad z_t = W_t E_{text}(x)
$$

$$
s(I,x)=\cos(z_v,z_t)
$$

$$
\mathcal{L}=\max\big(0,\, m - s(I,x^+) + s(I,x^-)\big)
$$

Es decir:
- primero se representan imagen y texto,
- luego se comparan en un espacio común,
- y finalmente se ajustan los parámetros para que los pares correctos queden más cerca que los incorrectos.

In [ ]:
def sample_negatives(y, num_classes=10):
    y_neg = y.clone()
    for i in range(len(y)):
        candidates = list(range(num_classes))
        candidates.remove(int(y[i]))
        y_neg[i] = np.random.choice(candidates)
    return y_neg

def margin_ranking_loss(img_z, pos_txt_z, neg_txt_z, margin=0.2):
    s_pos = F.cosine_similarity(img_z, pos_txt_z)
    s_neg = F.cosine_similarity(img_z, neg_txt_z)
    loss = torch.clamp(margin - s_pos + s_neg, min=0.0).mean()
    return loss, s_pos.mean().item(), s_neg.mean().item()

In [ ]:
def train_with_margin(model, loader, num_epochs=6, lr=1e-3, margin=0.2):
    for p in model.cnn_backbone.parameters():
        p.requires_grad = False

    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for xb, yb in loader:
            neg_y = sample_negatives(yb)

            img_z = model.encode_image(xb)
            pos_z = model.encode_text(yb)
            neg_z = model.encode_text(neg_y)

            loss, s_pos, s_neg = margin_ranking_loss(img_z, pos_z, neg_z, margin=margin)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item() * len(xb)

        print(f"Epoca {epoch+1:02d} | loss={total_loss/len(loader.dataset):.4f} | cos(pos)={s_pos:.4f} | cos(neg)={s_neg:.4f}")

train_with_margin(dual, align_train_loader, num_epochs=6, lr=1e-3, margin=0.2)

#### **10. Más avanzado: pérdida contrastiva con *in-batch negatives***

La **margin ranking loss** introduce la idea de separar pares positivos y negativos, pero normalmente compara un positivo con uno o pocos negativos explícitos.  
Una formulación más fuerte y más moderna consiste en usar **todo el lote (*batch*) como fuente de negativos**.

Supongamos un lote de $B$ pares imagen-texto correctamente alineados:

$$
\{(I_1,x_1), (I_2,x_2), \dots, (I_B,x_B)\}
$$

Después de codificar y proyectar cada modalidad, obtenemos:

$$
z^{(img)}_i \in \mathbb{R}^d, \qquad z^{(txt)}_j \in \mathbb{R}^d
$$

para $i,j=1,\dots,B$.

A partir de estos embeddings construimos una **matriz de similitud**:

$$
S_{ij} = \frac{{z^{(img)}_i}^{\top} z^{(txt)}_j}{\tau}
$$

donde:
- $S_{ij}$ mide la compatibilidad entre la imagen $I_i$ y el texto $x_j$,
- $\tau > 0$ es un parámetro de **temperatura**.

Si los embeddings están normalizados, entonces:

$$
S_{ij} = \frac{\cos\big(z^{(img)}_i, z^{(txt)}_j\big)}{\tau}
$$

##### **10.1 Interpretación de la matriz de similitud**

La matriz $S$ tiene una estructura muy importante:

- la **diagonal** $S_{ii}$ contiene los pares correctos imagen-texto,
- los elementos **fuera de la diagonal** $S_{ij}$, con $i \neq j$, actúan como **negativos implícitos** dentro del batch.

Esto significa que, en lugar de entrenar con un solo negativo por ejemplo, el modelo enfrenta simultáneamente muchos negativos:

- para una imagen $I_i$, todos los textos $x_j$ con $j \neq i$
- para un texto $x_i$, todas las imágenes $I_j$ con $j \neq i$.

Esa es precisamente la idea de **in-batch negatives**.

##### **10.2 Objetivo contrastivo tipo InfoNCE**

Para cada imagen $I_i$, queremos que su texto correcto $x_i$ tenga mayor similitud que los demás textos del lote.  
Eso se formula como una distribución sobre las columnas de la fila $i$:

$$
p(j \mid I_i)=
\frac{\exp(S_{ij})}{\sum_{k=1}^{B}\exp(S_{ik})}
$$

y se optimiza para maximizar la probabilidad del par correcto $j=i$.  
La pérdida correspondiente es:

$$
\mathcal{L}_{img \to txt}
=
-\frac{1}{B}\sum_{i=1}^{B}
\log
\frac{\exp(S_{ii})}{\sum_{j=1}^{B}\exp(S_{ij})}
$$

De forma simétrica, también puede definirse la pérdida texto -> imagen:

$$
\mathcal{L}_{txt \to img}
=
-\frac{1}{B}\sum_{i=1}^{B}
\log
\frac{\exp(S_{ii})}{\sum_{j=1}^{B}\exp(S_{ji})}
$$

En muchos modelos, incluida la formulación estilo CLIP, se usa la versión simétrica:

$$
\mathcal{L}
=
\frac{1}{2}
\left(
\mathcal{L}_{img \to txt}
+
\mathcal{L}_{txt \to img}
\right)
$$


##### **10.3 Papel de la temperatura $\tau$**

La temperatura $\tau$ controla la "agudeza" de la distribución softmax.

- Si $\tau$ es pequeña, pequeñas diferencias de similitud se amplifican mucho.
- Si $\tau$ es grande, la distribución se vuelve más suave.

Por tanto, $\tau$ regula cuán exigente es el contraste entre positivos y negativos.  
Desde una perspectiva geométrica, actúa como un factor de escala sobre la matriz de similitud.

##### **10.4 ¿Qué gana esta formulación frente a margin ranking?**

La pérdida contrastiva con in-batch negatives tiene varias ventajas:

**a. Usa muchos negativos a la vez**

No depende de muestrear un único negativo.  
Cada ejemplo del lote ve automáticamente muchos candidatos incorrectos.

**b. Introduce competencia global dentro del batch**

No basta con que el positivo sea "algo mejor" que un negativo aislado, debe destacar frente a muchas alternativas.

**c. Escala mejor para retrieval**
Esta formulación está muy bien alineada con tareas donde queremos que la imagen correcta recupere el texto correcto entre muchas opciones, o viceversa.

**d. Conecta directamente con modelos modernos**

La lógica de matriz de similitud + diagonal positiva + off-diagonal negativas es el núcleo de muchos modelos contrastivos actuales.

##### **10.5 Interpretación geométrica**

La loss contrastiva busca inducir una geometría en el espacio compartido donde:

- los pares correctos $(I_i, x_i)$ queden cerca
- los pares incorrectos $(I_i, x_j)$, $i \neq j$, queden relativamente lejos
- esa cercanía no se evalúe en forma aislada, sino frente a múltiples competidores simultáneamente.

En otras palabras, el objetivo ya no es solo:

$$
s(I,x^+) > s(I,x^-)
$$

sino algo más fuerte: que el par correcto sea el **más compatible dentro de un conjunto de alternativas**.

##### **10.6 Relación con CLIP**

Desde una perspectiva histórica, esta formulación puede verse como una extensión moderna del problema clásico de alineamiento visual-semántico:

1. representar imagen y texto,
2. proyectarlos a un espacio común,
3. definir una función de similitud,
4. optimizar para que los pares correctos se distingan de los incorrectos.

La diferencia es que ahora la separación se formula de manera **contrastiva y distribuida sobre el batch**, en vez de usar solo pérdidas de margen con negativos aislados.

Por eso, esta sección funciona bien como puente entre:
- los **visual-semantic embeddings** tempranos,
- y los modelos contrastivos modernos tipo **CLIP**.

##### **10.7 Resumen conceptual**

La lógica puede resumirse así:

$$
z^{(img)}_i = W_v E_{img}(I_i), \qquad
z^{(txt)}_j = W_t E_{text}(x_j)
$$

$$
S_{ij} = \frac{{z^{(img)}_i}^{\top} z^{(txt)}_j}{\tau}
$$

$$
\text{diagonal} = \text{pares positivos}, \qquad
\text{fuera de diagonal} = \text{negativos del batch}
$$

y el objetivo es que cada ejemplo asigne la mayor probabilidad posible a su par correspondiente.

In [ ]:
def clip_style_loss(img_z, txt_z, temperature=0.07):
    logits = img_z @ txt_z.T / temperature
    targets = torch.arange(len(img_z))

    loss_i = F.cross_entropy(logits, targets)
    loss_t = F.cross_entropy(logits.T, targets)
    return 0.5 * (loss_i + loss_t), logits

In [ ]:
dual_clip = DualEncoder(cnn, txt_vocab_size=10, txt_dim=32, shared_dim=32)

for p in dual_clip.cnn_backbone.parameters():
    p.requires_grad = False

opt = torch.optim.Adam(filter(lambda p: p.requires_grad, dual_clip.parameters()), lr=1e-3)

for epoch in range(6):
    dual_clip.train()
    total_loss = 0.0
    for xb, yb in align_train_loader:
        img_z = dual_clip.encode_image(xb)
        txt_z = dual_clip.encode_text(yb)

        loss, logits = clip_style_loss(img_z, txt_z, temperature=0.07)

        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(xb)

    print(f"Epoca {epoch+1:02d} | clip-style loss={total_loss/len(align_train_loader.dataset):.4f}")

#### **11. Evaluación: *retrieval* y Recall@K**

Una vez que imagen y texto han sido proyectados a un espacio compartido, el problema natural de evaluación no es únicamente clasificar, sino **recuperar el par correcto entre muchos candidatos**.

En este contexto, el modelo puede usarse en dos direcciones:

- **image-to-text retrieval**: dada una imagen, recuperar el texto correcto
- **text-to-image retrieval**: dado un texto, recuperar la imagen correcta.

Esto hace que la evaluación sea esencialmente un problema de **ranking**.

##### **11.1 Matriz de similitud**

Supongamos un conjunto de $N$ imágenes y $N$ textos alineados:

$$
\{(I_1,x_1), (I_2,x_2), \dots, (I_N,x_N)\}
$$

Una vez calculados sus embeddings compartidos, se construye una matriz de similitud:

$$
S \in \mathbb{R}^{N \times N}, \qquad S_{ij} = s(I_i, x_j)
$$

donde $S_{ij}$ mide qué tan compatible es la imagen $I_i$ con el texto $x_j$.

##### **Interpretación**

- La **diagonal** $S_{ii}$ contiene los pares correctos.
- Los valores fuera de la diagonal representan pares incorrectos o distractores.
- Evaluar retrieval consiste en verificar si el par correcto aparece entre los elementos de mayor similitud.

##### **11.2 Retrieval como problema de ranking**

Dada una imagen $I_i$, el modelo ordena todos los textos $x_1,\dots,x_N$ según sus similitudes:

$$
s(I_i, x_{(1)}) \ge s(I_i, x_{(2)}) \ge \cdots \ge s(I_i, x_{(N)})
$$

donde $(1), (2), \dots, (N)$ denotan el orden inducido por el ranking.

El objetivo ideal es que el texto correcto $x_i$ aparezca lo más arriba posible en esta lista.  
Análogamente, en text-to-image retrieval, queremos que la imagen correcta $I_i$ aparezca entre las más similares para el texto $x_i$.


##### **11.3 ¿Por qué no basta con accuracy?**

La **accuracy** tradicional supone un problema de clasificación cerrada, donde interesa una única predicción correcta por ejemplo.  
En retrieval, eso es insuficiente porque:

- el modelo produce un **ranking completo** de candidatos
- puede ser útil aunque el correcto no esté en primer lugar, pero sí entre los primeros
- la calidad del ranking importa más que una decisión binaria.

Por eso se usan métricas orientadas a recuperación, entre ellas **Recall@K**.

##### **11.4 Definición de Recall@K**

**Recall@K** mide la proporción de consultas para las cuales el elemento correcto aparece entre los primeros $K$ resultados del ranking.

Formalmente, para image-to-text retrieval:

$$
\mathrm{Recall@K}
=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}\big(\mathrm{rank}(x_i \mid I_i) \le K\big)
$$

donde:
- $\mathrm{rank}(x_i \mid I_i)$ es la posición del texto correcto $x_i$ en el ranking inducido por la imagen $I_i$,
- $\mathbf{1}(\cdot)$ es una función indicadora que vale 1 si la condición se cumple y 0 en caso contrario.

De forma análoga, para text-to-image retrieval:

$$
\mathrm{Recall@K}
=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}\big(\mathrm{rank}(I_i \mid x_i) \le K\big)
$$


##### **11.5 Interpretación de Recall@K**

- **Recall@1**: mide cuántas veces el par correcto fue el primero del ranking.  
  Es la forma más estricta de evaluación.

- **Recall@5**: mide cuántas veces el par correcto apareció entre los cinco primeros.  
  Es menos estricto y más informativo cuando hay muchos candidatos.

- **Recall@10**: útil cuando el espacio de búsqueda es grande y se quiere evaluar recuperación razonable aunque no perfecta.

##### **Ejemplo**

Si tenemos 100 consultas y en 72 de ellas el par correcto aparece entre los 5 primeros resultados, entonces:

$$
\mathrm{Recall@5} = 0.72
$$

Esto significa que el sistema recupera correctamente el objetivo dentro del top-5 en el 72% de los casos.

##### **11.6 Relación entre la matriz de similitud y Recall@K**

La matriz de similitud permite calcular directamente estas métricas:

- cada **fila** puede verse como una consulta imagen $\to$ textos
- cada **columna** puede verse como una consulta texto $\to$ imágenes.

Así, la evaluación consiste en ordenar cada fila o columna y verificar en qué posición queda el elemento correcto.

Esto conecta la evaluación con la propia formulación del entrenamiento contrastivo:
- si el modelo aprende bien, la diagonal de la matriz tenderá a dominar,
- si falla, los distractores pueden superar o mezclarse con los positivos.


##### **11.7 Qué información da Recall@K y qué no da**

Recall@K es muy útil porque:
- está alineado con la naturaleza del retrieval,
- es fácil de interpretar,
- permite comparar modelos de forma directa,
- fue ampliamente usado en trabajos de alineamiento visual-semántico.

Sin embargo, también tiene limitaciones:
- no dice **qué tan arriba** está el correcto si ya entra en el top-$K$,
- no distingue entre un correcto en posición 1 y otro en posición 5 cuando $K=5$,
- no ofrece una visión completa de todo el ranking.

Por eso, en evaluaciones más completas también pueden usarse métricas como:
- **median rank**,
- **mean rank**,
- **MRR** (*Mean Reciprocal Rank*),
- o **mAP** (*mean Average Precision*), dependiendo de la tarea.


##### **11.8 Relevancia**

Recall@K es especialmente importante en este curso porque muchos modelos tempranos de alineamiento visual-semántico fueron evaluados precisamente en tareas de:
- **image-to-text retrieval**,
- **text-to-image retrieval**.

Es decir, el problema no era solo "clasificar bien", sino aprender un espacio compartido en el que el par correcto quedara suficientemente alto en el ranking.

Por eso, esta métrica conecta directamente con:
- la similitud coseno,
- la margin ranking loss,
- y el paso posterior hacia pérdidas contrastivas modernas.

##### **11.9 Resumen conceptual**

La lógica de evaluación puede resumirse así:

$$
I_i \xrightarrow{E_{img}} z^{(img)}_i,
\qquad
x_j \xrightarrow{E_{text}} z^{(txt)}_j
$$

$$
S_{ij} = \operatorname{sim}\big(z^{(img)}_i, z^{(txt)}_j\big)
$$

$$
\text{rankear candidatos} \,\rightarrow\, \text{verificar posición del par correcto} \,\rightarrow\, \mathrm{Recall@K}
$$

En otras palabras, evaluar retrieval es evaluar si el espacio compartido aprendido organiza correctamente las correspondencias entre imagen y texto.

In [ ]:
def compute_similarity_matrix(model, loader):
    model.eval()
    all_img_z = []
    all_txt_z = []
    with torch.no_grad():
        for xb, yb in loader:
            all_img_z.append(model.encode_image(xb))
            all_txt_z.append(model.encode_text(yb))

    img_z = torch.cat(all_img_z, dim=0)
    txt_z = torch.cat(all_txt_z, dim=0)
    sims = img_z @ txt_z.T
    return sims.cpu().numpy()

def recall_at_k(sim_matrix, k=1):
    n = sim_matrix.shape[0]
    hits = 0
    for i in range(n):
        topk = np.argsort(-sim_matrix[i])[:k]
        if i in topk:
            hits += 1
    return hits / n

sim_margin = compute_similarity_matrix(dual, align_test_loader)
sim_clip = compute_similarity_matrix(dual_clip, align_test_loader)

print("Margin model | Recall@1:", round(recall_at_k(sim_margin, k=1), 4), "| Recall@5:", round(recall_at_k(sim_margin, k=5), 4))
print("CLIP-style   | Recall@1:", round(recall_at_k(sim_clip, k=1), 4), "| Recall@5:", round(recall_at_k(sim_clip, k=5), 4))

In [ ]:
subset = 12
plt.figure(figsize=(6, 5))
plt.imshow(sim_clip[:subset, :subset], aspect="auto")
plt.colorbar()
plt.title("Matriz de similitud imagen-texto (modelo clip-style)")
plt.xlabel("textos")
plt.ylabel("imágenes")
plt.tight_layout()
plt.show()

#### **12. Comparación conceptual: *margin ranking* vs *InfoNCE***

Tanto la **margin ranking loss** como la **InfoNCE / CLIP-style loss** buscan resolver el mismo problema de fondo:

> aprender un espacio compartido en el que pares imagen-texto compatibles queden cerca y pares incompatibles queden lejos.

Sin embargo, difieren de forma importante en:
- cómo definen los negativos,
- cómo estructuran el objetivo de entrenamiento,
- y qué tipo de geometría inducen en el espacio compartido.

##### **12.1 Punto en común: el problema de alineamiento**

En ambos casos partimos de embeddings proyectados:

$$
z_v = W_v E_{img}(I), \qquad z_t = W_t E_{text}(x)
$$

y de una función de similitud, típicamente:

$$
s(I,x)=\cos(z_v,z_t)
$$

La diferencia no está en la existencia del espacio compartido, sino en **cómo se fuerza matemáticamente la separación entre positivos y negativos**.

##### **12.2 Margin ranking: comparación local entre positivos y negativos explícitos**

La formulación clásica de ranking con margen se basa en tripletas o comparaciones locales del tipo:

- una imagen $I$,
- un texto correcto $x^+$,
- un texto incorrecto $x^-$.

La pérdida típica es:

$$
\mathcal{L}
=
\max\big(0,\, m - s(I,x^+) + s(I,x^-)\big)
$$

donde $m$ es el margen deseado.

##### **Interpretación**
La lógica es simple:
- el par positivo debe ser más similar que el negativo,
- además, debe superarlo por al menos un margen $m$.

Esto convierte el aprendizaje en un problema de **orden relativo local**:  
para cada ejemplo, el modelo solo necesita garantizar una separación suficiente entre ciertos pares concretos.

##### **Ventajas**
- es conceptualmente simple,
- resulta fácil de interpretar geométricamente,
- fue muy natural para tareas tempranas de retrieval,
- encaja bien con formulaciones de visual-semantic embeddings.

##### **Limitaciones**
- depende fuertemente de cómo se eligen los negativos,
- si los negativos son demasiado fáciles, el aprendizaje puede volverse poco informativo,
- no aprovecha toda la estructura de similitudes disponible en un lote completo,
- puede requerir estrategias de *hard negative mining* para ser realmente efectivo.

En otras palabras, **margin ranking** organiza el espacio multimodal mediante comparaciones locales y explícitas.

##### **12.3 InfoNCE / CLIP-style: competencia global dentro del batch**

La pérdida contrastiva tipo **InfoNCE** parte de una idea distinta:  
en vez de comparar un positivo con uno o pocos negativos seleccionados explícitamente, utiliza **todos los demás elementos del lote como negativos implícitos**.

Se construye una matriz de similitud:

$$
S_{ij}=
\frac{{z^{(img)}_i}^{\top} z^{(txt)}_j}{\tau}
$$

donde la diagonal $S_{ii}$ contiene los pares correctos y los términos fuera de la diagonal actúan como distractores.

La loss imagen -> texto es:

$$
\mathcal{L}_{img\to txt}
=
-\frac{1}{B}\sum_{i=1}^{B}
\log
\frac{\exp(S_{ii})}{\sum_{j=1}^{B}\exp(S_{ij})}
$$

y de forma simétrica se define texto -> imagen.  
La versión estilo CLIP promedia ambas direcciones.

##### **Interpretación**
Aquí el modelo no solo debe hacer que el positivo sea “mejor que un negativo”, sino que debe volverlo el **más compatible entre muchas alternativas simultáneas**.

Esto convierte el problema en una especie de clasificación contrastiva dentro del lote:
- para cada imagen, identificar cuál es su texto correcto entre todos los textos del batch
- para cada texto, identificar cuál es su imagen correcta entre todas las imágenes del batch.

##### **Ventajas**
- usa muchos negativos de forma natural,
- induce una competencia más fuerte entre candidatos,
- se alinea muy bien con tareas de retrieval,
- escala mejor cuando se dispone de grandes lotes o grandes colecciones de pares.

##### **Limitaciones**
- depende del tamaño y composición del lote,
- puede requerir más memoria y cómputo,
- su comportamiento está influido por la temperatura $\tau$,
- la calidad de los negativos depende implícitamente del batch.

En otras palabras, **InfoNCE** organiza el espacio multimodal mediante una **competencia global distribuida sobre el lote**.

##### **12.4 Diferencia conceptual central**

La diferencia más importante entre ambos enfoques puede resumirse así:

- **Margin ranking** aprende a separar el positivo de uno o algunos negativos explícitos.
- **InfoNCE / CLIP-style** aprende a distinguir el positivo frente a todos los candidatos del lote al mismo tiempo.

O dicho de otro modo:

$$
\text{margin ranking} \approx \text{comparación local}
$$

$$
\text{InfoNCE} \approx \text{discriminación global dentro del batch}
$$

Esta diferencia tiene implicancias geométricas importantes:

- en margin ranking, la estructura del espacio se construye a partir de restricciones locales
- en InfoNCE, el espacio se organiza más directamente como un problema de ranking global.

##### **12.5 Relación histórica entre ambos**

Desde una perspectiva histórica, ambos enfoques pertenecen a la misma familia amplia de problemas:  
**alinear modalidades heterogéneas en un espacio vectorial común**.

Lo que cambia entre ellos es el refinamiento del objetivo:

1. **Primera etapa:**  
   embeddings visual-semánticos con pérdidas de ranking o margen.

2. **Etapa posterior:**  
   objetivos contrastivos con muchos negativos, lotes grandes y entrenamiento a escala.

##### **12.7 Resumen conceptual**

Para fines iniciales, lo esencial no es aprender todos los detalles de CLIP, sino entender que:

- ambos enfoques suponen **embeddings imagen-texto en un espacio compartido**
- ambos usan una función de similitud para medir compatibilidad
- ambos intentan acercar positivos y alejar negativos
- la diferencia principal está en la **forma de definir y explotar los negativos**.

Por eso, esta comparación funciona como un excelente puente entre:
- los modelos tempranos de alineamiento visual-semántico,
- y el bloque posterior del curso sobre aprendizaje contrastivo.

#### **13. Preguntas de repaso**

1. ¿Qué hipótesis representacional comparten Bag of Words en texto y BoVW en visión?  
2. ¿Qué información espacial se pierde en BoVW y cómo podría recuperarse parcialmente?  
3. ¿Qué cambia conceptualmente al pasar de una representación global a una representación por regiones?  
4. ¿Por qué la normalización de embeddings vuelve especialmente natural el uso de similitud coseno?  
5. ¿Qué diferencia teórica hay entre separar un positivo de un solo negativo y separarlo de todos los negativos del batch?  
6. ¿Por qué Recall@K es una métrica más adecuada que accuracy para tareas de retrieval?  
7. ¿Qué partes de este cuaderno conectan directamente con DeViSE, Karpathy & Li o con el tránsito posterior hacia CLIP?

In [ ]:
### Tus respuestas
1. Bag of Words y BoVW comparten la idea de representar un objeto mediante la frecuencia de elementos discretos, ignorando inicialmente su orden.

2. BoVW pierde la ubicación de los elementos visuales; esta información puede recuperarse parcialmente mediante particiones espaciales o regiones.

3. Una representación global resume toda la imagen en un único vector, mientras que una representación por regiones conserva información local de distintas partes de la imagen.

4. Al normalizar embeddings, la comparación depende principalmente de la dirección de los vectores, haciendo que la similitud coseno sea una medida natural de cercanía semántica.

5. Separar un positivo de un solo negativo es una tarea más simple; separarlo de todos los negativos del batch produce representaciones más discriminativas.

6. Recall@K es más adecuado porque evalúa si la respuesta correcta aparece entre los primeros resultados recuperados, que es el objetivo principal del retrieval.

7. El uso de embeddings compartidos, similitud coseno y recuperación multimodal conecta directamente con DeViSE, Karpathy & Li y posteriormente con CLIP.
